# CoWork Enterprise Demo — Govern AI Access
## Notebook 02 — Govern AI Access (Phase 2)

_Icons used throughout: 🛠️ Action  📌 Note  🔹 Info_

> ⚠️ _Generated from `build_notebooks.py` — edit the builder and re-run. Direct edits to this notebook are overwritten._

---

### What This Notebook Does:

1. 🛠️ Creates a **CoWork-only consumer role** for business users
2. 🛠️ Grants it the **four independent layers** of access an agent needs at runtime
3. 🛠️ Attaches an **`AI_REDACT` masking policy** and proves it applies to agent output automatically

---

### Why Governance Is Inherited, Not Configured Separately:

🔹 **CoWork has no separate security layer.** Every query runs as the authenticated user, inheriting their role, masking, and row-access policies. Getting RBAC right *is* your CoWork security posture.

🔹 Cortex Agents run with **caller's rights** — each tool uses the *calling user's* privileges, not the owner's. So the consumer role needs access to every object the agent touches:

| Layer | What it grants | Privilege |
|---|---|---|
| 1. Platform | Can call Cortex + use compute | `CORTEX_USER` + `USAGE` on DB/WH |
| 2. Which agent | See/use the specific agent | `USAGE ON AGENT` (granted in NB03) |
| 3. Tool/resource | The search service + semantic view | `USAGE` on each |
| 4. Data | The underlying tables | `SELECT` (+ masking/row-access still applies) |

> **Documentation:** [Agent access control](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents#access-control-requirements)

---

### Estimated Time: **15–20 minutes**

### Prerequisites:
- Notebook 01 complete (semantic view + search service created)

## Step 1: Set Context

In [ ]:
%%sql -r set_context
USE ROLE ACCOUNTADMIN;
USE DATABASE COWORK_ENTERPRISE_DEMO;
USE WAREHOUSE COWORK_ENTERPRISE_DEMO_WH;


## Step 2: Create the CoWork-Only Role + Four-Layer Grants

🛠️ One role, `COWORK_ENTERPRISE_DEMO_SI_USER`, gets layers 1, 3, and 4 now. Layer 2 (`USAGE ON AGENT`) is deferred to Notebook 03 because the agent doesn't exist yet.

🔹 Pairing this role with `ALLOWED_INTERFACES = ('SNOWFLAKE_INTELLIGENCE')` on the user object confines a business user to the CoWork chat — no worksheets, no SQL — we provision exactly such a user at go-live (NB05).

In [ ]:
%%sql -r four_layer_grants
USE ROLE ACCOUNTADMIN;
CREATE ROLE IF NOT EXISTS COWORK_ENTERPRISE_DEMO_SI_USER COMMENT = 'Business users who access the demo agent via CoWork only';
-- Tier 1: platform entitlement (can use Cortex + compute)
GRANT USAGE ON DATABASE COWORK_ENTERPRISE_DEMO TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
GRANT USAGE ON WAREHOUSE COWORK_ENTERPRISE_DEMO_WH TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
-- Tier 3: tool / resource access (search service + semantic view)
GRANT USAGE ON SCHEMA COWORK_ENTERPRISE_DEMO.AGENTS TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
GRANT USAGE ON CORTEX SEARCH SERVICE COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_SEARCH TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
GRANT USAGE ON CORTEX SEARCH SERVICE COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_CLIENT_SEARCH TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
GRANT USAGE ON SCHEMA COWORK_ENTERPRISE_DEMO.SEMANTIC TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
GRANT USAGE ON SEMANTIC VIEW COWORK_ENTERPRISE_DEMO.SEMANTIC.DEMO_SV TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
-- Tier 4: data access (masking / row-access still applies automatically)
GRANT USAGE ON SCHEMA COWORK_ENTERPRISE_DEMO.ANALYTICS TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
GRANT SELECT ON ALL TABLES IN SCHEMA COWORK_ENTERPRISE_DEMO.ANALYTICS TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
-- Tier 2 (USAGE ON AGENT) is granted in Notebook 03, after the agent exists.
SELECT 'SI role created with tiers 1, 3, 4' AS STATUS;


### 📌 Reference only — do NOT run on a shared account
For a controlled enterprise rollout you would restrict Cortex to intended roles. We skip this here because `CORTEX_USER` is granted to PUBLIC and shared with other teams.

```sql
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE PUBLIC;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;
```
🔹 `CORTEX_AGENT_USER` grants only the Cortex Agents API (enough for CoWork), not every Cortex feature.

### 🔹 What the Agent's SQL Can Reach (and How to Scope It)

Since **April 2026**, Cortex Agents generate SQL **directly** rather than delegating to a separate Analyst service. Practically:

- **Can access:** any table the **caller's role** can `SELECT` — not only the tables in the semantic view.
- **Still constrained by:** RBAC (no `SELECT`, no data), the semantic view (dimensions / metrics / verified queries steer the generated SQL), and orchestration instructions.

📌 **For strict scoping** (agent may touch *only* semantic-view tables), grant the consumer role `SELECT` on **just those tables** — not broad schema-level `SELECT` — and optionally add an orchestration instruction: *"Only query tables defined in the semantic view."* The `SI_USER` role here is scoped to the demo schema; tighten it to specific tables for a locked-down rollout.

> **Reference:** [Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents) — see the April 2026 SQL-generation release note.

## Step 3: Enforce PII at the Data Layer (`AI_REDACT`)

🛠️ Attach a masking policy that redacts email for non-privileged roles. Because governance is inherited, **any agent answer that surfaces this column is masked the same way**.

📌 **Enforce PII at the data layer, never in prompt instructions** — a prompt rule can be talked around; a masking policy cannot. `AI_REDACT` detects and redacts sensitive entities (here, EMAIL).

In [ ]:
%%sql -r create_masking_policy
CREATE MASKING POLICY IF NOT EXISTS COWORK_ENTERPRISE_DEMO.ANALYTICS.EMAIL_REDACT AS (val STRING)
  RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() IN ('COWORK_ENTERPRISE_DEMO_ADMIN','ACCOUNTADMIN') THEN val
    ELSE AI_REDACT(val, ['EMAIL'])
  END;

ALTER TABLE COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS
  MODIFY COLUMN EMAIL SET MASKING POLICY COWORK_ENTERPRISE_DEMO.ANALYTICS.EMAIL_REDACT;

SELECT 'Masking policy applied to CLIENTS.EMAIL' AS STATUS;


### 📌 Why a Masking Policy — Not Prompt Instructions

A common question: *"Can't I just tell the agent not to show PII in its instructions?"* **No — orchestration instructions are not a security boundary.** They steer the LLM but aren't guaranteed under every input; a creative prompt can talk around them. Enforce PII at the **data layer**, where the agent's SQL only ever sees masked values:

| Layer | What it does | Bypassable by a prompt? |
|---|---|---|
| Orchestration instruction ("don't show PII") | LLM guidance only | **Yes** |
| RBAC (role can't `SELECT` the column) | Engine-enforced | **No** |
| Dynamic Data Masking (`AI_REDACT`) | Redacts at the data layer before results return | **No** |

🔹 That's why the policy above lives on the column, not in the agent spec — the agent never receives the raw value, no matter what the user asks.

## Step 4: Verify

📌 Confirm the role has the expected grants and the masking policy is active.

In [ ]:
%%sql -r verify_grants
SHOW GRANTS TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;


In [ ]:
%%sql -r verify_masking
SELECT EMAIL FROM COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS LIMIT 5;


## ✅ Notebook 02 Complete

### What You Just Built:
- CoWork-only role `COWORK_ENTERPRISE_DEMO_SI_USER` with the four-layer access model (agent grant follows in NB03)
- `AI_REDACT` masking policy on `CLIENTS.EMAIL`, applied at the data layer

---

### Key Points:
- CoWork security = your RBAC. There is no separate AI security layer to configure.
- Agents run with caller's rights, so grant the consumer role every object the agent touches.
- Masking and row-access policies flow through to agent output automatically — enforce PII in the data, not the prompt.

---

### Next:

Continue to **Notebook 03 — Build & Harden the Agent**: create the agent from a specification, grant the final access layer, and confirm account guardrails.